# CPE028 Device Repair Assessment — Full Model Training (Kaggle)

This notebook trains all 5 models for the device repair assessment system.
Adapted from the Google Colab version for use on Kaggle.

| # | Model | Type | Framework |
|---|-------|------|-----------|
| 1 | Issue Classifier | Text (6 classes) | sklearn VotingClassifier |
| 2 | Repairability Scorer | Text (regression) | sklearn VotingRegressor |
| 3 | Crack Detector | Image (binary) | ResNet18 (PyTorch) |
| 4 | Corrosion Detector | Image (5 classes) | ResNet18 (PyTorch) |
| 5 | Component Classifier | Image (10 classes) | ResNet18 (PyTorch) |

---
## 0. Setup

In [1]:
import os, json, re, glob, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms, datasets
from pathlib import Path
from tqdm import tqdm
from PIL import Image
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (RandomForestClassifier, VotingClassifier,
                              RandomForestRegressor, VotingRegressor)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, mean_absolute_error,
                             r2_score, ConfusionMatrixDisplay,
                             precision_recall_fscore_support, roc_curve, auc)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, label_binarize
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeRegressor
import joblib

warnings.filterwarnings("ignore")

# === Kaggle Dataset Paths ===
KAGGLE_DATA = Path('/kaggle/input/datasets/leonpascual1/cpe028-dataset/datasets')

# Text datasets
ISSUE_CSV = KAGGLE_DATA / 'text/pre_processed/processed_issue_dataset.csv'
REPAIRABILITY_CSV = KAGGLE_DATA / 'text/pre_processed/processed_repairability_dataset1.csv'

# Image datasets
CRACK_DATA = KAGGLE_DATA / 'image/54670865_Data Set_Smartphone Screen Damage Detection/project_data'
CORROSION_DATA = KAGGLE_DATA / 'image/DATA_SET_FOR_RELEASE_CORROSION/DATA_SET_FOR_RELEASE/renamed'
COMPONENT_DATA = KAGGLE_DATA / 'image/Laptop Components Image Dataset to Classify Different Components/Augmented Data/Augmented Data'

# Output directory (Kaggle writable)
OUTPUT_DIR = Path('/kaggle/working')
RESULTS_DIR = OUTPUT_DIR / 'training/results'
MODEL_DIR = OUTPUT_DIR / 'models'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Data root:  {KAGGLE_DATA}")
print(f"Output dir: {OUTPUT_DIR}")

Device: cuda
GPU: Tesla T4
Data root:  /kaggle/input/datasets/leonpascual1/cpe028-dataset/datasets
Output dir: /kaggle/working


---
## 0b. Visualization Functions

In [2]:
def plot_confusion_matrix(y_true, y_pred, classes, save_path, title="Confusion Matrix"):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(max(6, len(classes)), max(5, len(classes) - 1)))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
    disp.plot(ax=ax, cmap="Blues", values_format="d")
    ax.set_title(title)
    plt.tight_layout()
    Path(save_path).parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(save_path, dpi=150)
    plt.show()
    plt.close(fig)

def plot_training_curves(train_losses, val_losses, train_accs, val_accs, save_path):
    epochs = range(1, len(train_losses) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    ax1.plot(epochs, train_losses, "o-", label="Train")
    if val_losses:
        ax1.plot(epochs, val_losses, "s-", label="Validation")
    ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss"); ax1.set_title("Training Loss")
    ax1.legend(); ax1.grid(True, alpha=0.3)
    ax2.plot(epochs, train_accs, "o-", label="Train")
    if val_accs:
        ax2.plot(epochs, val_accs, "s-", label="Validation")
    ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy"); ax2.set_title("Training Accuracy")
    ax2.legend(); ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    Path(save_path).parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(save_path, dpi=150)
    plt.show()
    plt.close(fig)

def plot_roc_auc(y_true, y_scores, classes, save_path, title="ROC Curve"):
    n_classes = len(classes)
    y_true_bin = label_binarize(y_true, classes=list(range(n_classes)))
    fig, ax = plt.subplots(figsize=(8, 6))
    if n_classes == 2:
        fpr, tpr, _ = roc_curve(y_true, y_scores[:, 1])
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, linewidth=2, label=f"AUC = {roc_auc:.3f}")
    else:
        colors = plt.cm.Set1(np.linspace(0, 1, n_classes))
        for i, (cls, color) in enumerate(zip(classes, colors)):
            fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_scores[:, i])
            roc_auc = auc(fpr, tpr)
            ax.plot(fpr, tpr, color=color, linewidth=2, label=f"{cls} (AUC = {roc_auc:.3f})")
    ax.plot([0, 1], [0, 1], "k--", linewidth=1)
    ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
    ax.set_title(title); ax.legend(loc="lower right"); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    Path(save_path).parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(save_path, dpi=150)
    plt.show()
    plt.close(fig)

def plot_per_class_pr(y_true, y_pred, classes, save_path, title="Per-Class Precision & Recall"):
    precision, recall, _, _ = precision_recall_fscore_support(y_true, y_pred, labels=classes)
    x = np.arange(len(classes))
    width = 0.35
    fig, ax = plt.subplots(figsize=(max(8, len(classes) * 1.2), 5))
    bars1 = ax.bar(x - width / 2, precision, width, label="Precision", color="#2196F3")
    bars2 = ax.bar(x + width / 2, recall, width, label="Recall", color="#FF9800")
    ax.set_xlabel("Class"); ax.set_ylabel("Score"); ax.set_title(title)
    ax.set_xticks(x)
    ax.set_xticklabels(classes, rotation=45, ha="right")
    ax.set_ylim(0, 1.1); ax.legend(); ax.grid(True, alpha=0.3, axis="y")
    for bar in bars1:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
                f"{bar.get_height():.2f}", ha="center", va="bottom", fontsize=8)
    for bar in bars2:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
                f"{bar.get_height():.2f}", ha="center", va="bottom", fontsize=8)
    plt.tight_layout()
    Path(save_path).parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(save_path, dpi=150)
    plt.show()
    plt.close(fig)

def plot_regression_results(y_true, y_pred, save_path, title="Regression: Predicted vs Actual"):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    residuals = y_true - y_pred
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    ax1.scatter(y_true, y_pred, alpha=0.5, edgecolors="k", linewidth=0.5)
    mn, mx = min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())
    ax1.plot([mn, mx], [mn, mx], "r--", linewidth=2, label="Perfect")
    ax1.set_xlabel("Actual"); ax1.set_ylabel("Predicted"); ax1.set_title(title)
    ax1.legend(); ax1.grid(True, alpha=0.3)
    ax2.hist(residuals, bins=30, edgecolor="black", alpha=0.7)
    ax2.axvline(0, color="r", linestyle="--", linewidth=2)
    ax2.set_xlabel("Residual (Actual - Predicted)"); ax2.set_ylabel("Count")
    ax2.set_title("Residual Distribution"); ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    Path(save_path).parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(save_path, dpi=150)
    plt.show()
    plt.close(fig)

def img_train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in tqdm(loader, desc="Training"):
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / len(loader), correct / total

def img_val_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            preds = outputs.argmax(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return total_loss / len(loader), correct / total, all_preds, all_labels

print("Visualization functions and training helpers loaded.")

Visualization functions and training helpers loaded.


---
## 1. Issue Classifier (NLP — Text)

VotingClassifier with LinearSVC + RandomForest + LogisticRegression

Dataset: `processed_issue_dataset.csv`

In [3]:
def clean_text(value):
    if pd.isna(value): return ""
    text = str(value).lower().replace("\n", " ")
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()

# Load preprocessed dataset
df = pd.read_csv(ISSUE_CSV)
df = df[["text", "text_clean", "issue_label", "source"]].dropna(subset=["text_clean"])
df = df[df["issue_label"].astype(str).str.strip() != ""]
print(f"Issue dataset: {len(df)} samples")
print(df["issue_label"].value_counts())

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    df[["text_clean", "source"]], df["issue_label"],
    test_size=0.2, random_state=42, stratify=df["issue_label"]
)

# Build pipeline
pipeline = Pipeline([
    ("preprocess", ColumnTransformer([
        ("text", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=200), "text_clean"),
        ("source", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["source"]),
    ], remainder="drop")),
    ("model", VotingClassifier([
        ("svc", LinearSVC(C=1.0, class_weight="balanced", max_iter=1000)),
        ("rf", RandomForestClassifier(n_estimators=100, random_state=42,
                                       class_weight="balanced_subsample", n_jobs=-1)),
        ("logreg", LogisticRegression(max_iter=2000, class_weight="balanced")),
    ], voting="hard")),
])

pipeline.fit(X_train, y_train)
pred = pipeline.predict(X_test)
acc = accuracy_score(y_test, pred)
print(f"\nAccuracy: {acc:.4f}")
print(classification_report(y_test, pred))

# Save model
joblib.dump(pipeline, MODEL_DIR / "issue_classifier_voting.joblib")

# Visualizations
classes = sorted(set(y_test))
plot_confusion_matrix(y_test, pred, classes,
    RESULTS_DIR / "issue_classifier_confusion_matrix.png", "Issue Classifier — Confusion Matrix")
plot_per_class_pr(y_test, pred, classes,
    RESULTS_DIR / "issue_classifier_per_class_pr.png", "Issue Classifier — Per-Class Precision & Recall")

# Save results JSON
report = classification_report(y_test, pred, output_dict=True)
results = {
    "issue_model": {
        "accuracy": round(float(acc), 4),
        "classification_report": {
            k: round(v, 4) if isinstance(v, float) else v
            for k, v in report.items() if k not in ("macro avg", "weighted avg")
        },
    },
    "datasets_used": ["processed_issue_dataset.csv"],
    "train_size": len(X_train),
    "test_size": len(X_test),
    "test_split_ratio": 0.2,
    "total_samples": len(df),
    "saved_model": str(MODEL_DIR / "issue_classifier_voting.joblib"),
}
(RESULTS_DIR / "issue_classifier_results.json").write_text(json.dumps(results, indent=2))
print("\nDone: Issue classifier.")

Issue dataset: 4000 samples
issue_label
Software issue         951
Hardware issue         877
Battery degradation    816
Cracked screen         721
Water/Liquid damage    521
Unknown                114
Name: count, dtype: int64

Accuracy: 0.9725
                     precision    recall  f1-score   support

Battery degradation       0.98      0.98      0.98       163
     Cracked screen       0.92      1.00      0.96       144
     Hardware issue       1.00      1.00      1.00       176
     Software issue       1.00      0.92      0.96       190
            Unknown       0.78      0.91      0.84        23
Water/Liquid damage       1.00      1.00      1.00       104

           accuracy                           0.97       800
          macro avg       0.95      0.97      0.96       800
       weighted avg       0.97      0.97      0.97       800


Done: Issue classifier.


---
## 2. Repairability Scorer (NLP — Regression)

VotingRegressor with DecisionTree + RandomForest

Dataset: `processed_repairability_dataset1.csv`

In [4]:
# Load preprocessed dataset
model_df = pd.read_csv(REPAIRABILITY_CSV)
model_df["repairability_score"] = pd.to_numeric(model_df["repairability_score"], errors="coerce")
model_df = model_df.dropna(subset=["repairability_score"])

# Numeric features
for col in ["repair_cost", "customer_rating", "usage_duration", "price", "Img Score"]:
    model_df[col] = pd.to_numeric(model_df.get(col, 0), errors="coerce").fillna(0)

# Frequency encode categoricals
for col in ["brand", "category", "failure_type", "warranty_status"]:
    model_df[col] = model_df[col].fillna("unknown").astype(str)
    freq = model_df[col].value_counts(normalize=True)
    model_df[col + "_freq"] = model_df[col].map(freq).astype(float)

# Label encode categoricals
for col in ["brand", "category", "failure_type", "warranty_status", "source"]:
    le = LabelEncoder()
    model_df[col + "_le"] = le.fit_transform(model_df[col].fillna("unknown"))

# Data availability flags
model_df["has_repair_data"] = (model_df["repair_cost"] > 0).astype(int)
model_df["has_price"] = (model_df["price"] > 0).astype(int)

print(f"Repairability dataset: {len(model_df)} samples")
print(model_df["source"].value_counts())

NUM_FEATURES = ["repair_cost", "customer_rating", "usage_duration", "price", "Img Score",
                "brand_freq", "category_freq", "failure_type_freq", "warranty_status_freq",
                "brand_le", "category_le", "failure_type_le", "warranty_status_le", "source_le",
                "has_repair_data", "has_price"]

X = model_df[NUM_FEATURES]
y = model_df["repairability_score"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipeline = Pipeline([
    ("model", VotingRegressor([
        ("dt", DecisionTreeRegressor(max_depth=10, random_state=42)),
        ("rf", RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)),
    ])),
])

pipeline.fit(X_train, y_train)
pred = pipeline.predict(X_test)
mae = mean_absolute_error(y_test, pred)
r2 = r2_score(y_test, pred)
print(f"MAE: {mae:.4f} | R2: {r2:.4f}")

# Save model
joblib.dump(pipeline, MODEL_DIR / "repairability_voting_regressor.joblib")

# Visualization
plot_regression_results(y_test.tolist(), pred.tolist(),
    RESULTS_DIR / "repairability_scorer_regression.png", "Repairability Scorer — Predicted vs Actual")

# Save results JSON
results = {
    "repairability_model": {"mae": round(float(mae), 4), "r2": round(float(r2), 4)},
    "datasets_used": ["processed_repairability_dataset1.csv"],
    "features_used": NUM_FEATURES,
    "train_size": len(X_train),
    "test_size": len(X_test),
    "test_split_ratio": 0.2,
    "total_samples": len(model_df),
    "saved_model": str(MODEL_DIR / "repairability_voting_regressor.joblib"),
}
(RESULTS_DIR / "repairability_scorer_results.json").write_text(json.dumps(results, indent=2))
print("\nDone: Repairability scorer.")

Repairability dataset: 6548 samples
source
tech-gadget-failures    3000
laptop-dataset          2000
repairability-final      865
repair-history           528
ifixit                   155
Name: count, dtype: int64
MAE: 0.2656 | R2: 0.7448

Done: Repairability scorer.


---
## 2b. Text Models Complete

Both text models are trained and saved. To download them:
1. Click **Output** in the right sidebar
2. Find `training/results/` and `models/` folders
3. Click the download icon next to each file

Files saved:
- `models/issue_classifier_voting.joblib`
- `models/repairability_hist_gradient_boosting.joblib`
- `training/results/*.json` and `*.png`

---
## 3. Crack Detector (Image — Binary)

ResNet18 binary classifier: `cracked` vs `not_cracked`

Uses light augmentation + dropout to prevent overfitting on small dataset.

Dataset: Smartphone Screen Damage Detection

In [5]:
# === Crack Detector: per-model config ===
CRACK_IMG_SIZE = 224
CRACK_BATCH_SIZE = 32
CRACK_NUM_EPOCHS = 15
CRACK_LR = 0.0005
CRACK_WEIGHT_DECAY = 1e-3
CRACK_PATIENCE = 3

# Light augmentation only — cracks are orientation-sensitive
crack_train_transform = transforms.Compose([
    transforms.Resize((CRACK_IMG_SIZE, CRACK_IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

crack_val_transform = transforms.Compose([
    transforms.Resize((CRACK_IMG_SIZE, CRACK_IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class CrackDetector(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.backbone = models.resnet18(weights="DEFAULT")
        self.backbone.fc = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(self.backbone.fc.in_features, num_classes),
        )
    def forward(self, x): return self.backbone(x)

# Load datasets
CRACK_ROOT = CRACK_DATA
CLASSES_CRACK = ["not_cracked", "cracked"]

train_ds = datasets.ImageFolder(str(CRACK_ROOT / "train"), transform=crack_train_transform)
valid_ds = datasets.ImageFolder(str(CRACK_ROOT / "valid"), transform=crack_val_transform) if (CRACK_ROOT / "valid").exists() else None
test_ds = datasets.ImageFolder(str(CRACK_ROOT / "test"), transform=crack_val_transform) if (CRACK_ROOT / "test").exists() else None
print(f"Crack — Train: {len(train_ds)} | Valid: {len(valid_ds) if valid_ds else 0} | Test: {len(test_ds) if test_ds else 0}")

train_loader = DataLoader(train_ds, batch_size=CRACK_BATCH_SIZE, shuffle=True, num_workers=0)
valid_loader = DataLoader(valid_ds, batch_size=CRACK_BATCH_SIZE, shuffle=False, num_workers=0) if valid_ds else None
test_loader = DataLoader(test_ds, batch_size=CRACK_BATCH_SIZE, shuffle=False, num_workers=0) if test_ds else None

model = CrackDetector().to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=CRACK_LR, weight_decay=CRACK_WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CRACK_NUM_EPOCHS)

best_acc = 0.0
patience_counter = 0
t_losses, t_accs, v_losses, v_accs = [], [], [], []

for epoch in range(CRACK_NUM_EPOCHS):
    tl, ta = img_train_epoch(model, train_loader, criterion, optimizer, DEVICE)
    t_losses.append(tl); t_accs.append(ta)
    msg = f"Epoch {epoch+1}/{CRACK_NUM_EPOCHS} — Train Loss: {tl:.4f} Acc: {ta:.4f}"
    if valid_loader:
        vl, va, _, _ = img_val_epoch(model, valid_loader, criterion, DEVICE)
        v_losses.append(vl); v_accs.append(va)
        msg += f" | Val Loss: {vl:.4f} Acc: {va:.4f}"
        if va > best_acc:
            best_acc = va
            patience_counter = 0
            torch.save(model.state_dict(), MODEL_DIR / "crack_detector.pth")
        else:
            patience_counter += 1
    print(msg)
    scheduler.step()
    if patience_counter >= CRACK_PATIENCE:
        print(f"Early stopping at epoch {epoch+1}")
        break

# Load best and evaluate on test set
model.load_state_dict(torch.load(MODEL_DIR / "crack_detector.pth", map_location=DEVICE, weights_only=True))
if test_loader:
    _, test_acc, test_preds, test_labels = img_val_epoch(model, test_loader, criterion, DEVICE)
    print(f"\nTest Accuracy: {test_acc:.4f}")
    print(classification_report(test_labels, test_preds, target_names=CLASSES_CRACK))

    plot_confusion_matrix(test_labels, test_preds, CLASSES_CRACK,
        RESULTS_DIR / "crack_detector_confusion_matrix.png", "Crack Detector — Confusion Matrix")
    plot_training_curves(t_losses, v_losses, t_accs, v_accs,
        RESULTS_DIR / "crack_detector_training_curves.png")
    plot_per_class_pr(test_labels, test_preds, CLASSES_CRACK,
        RESULTS_DIR / "crack_detector_per_class_pr.png", "Crack Detector — Per-Class P&R")

    model.eval()
    all_scores = []
    with torch.no_grad():
        for images, _ in test_loader:
            all_scores.extend(torch.softmax(model(images.to(DEVICE)), dim=1).cpu().numpy())
    plot_roc_auc(test_labels, np.array(all_scores), CLASSES_CRACK,
        RESULTS_DIR / "crack_detector_roc_auc.png", "Crack Detector — ROC Curve")

results = {
    "model_type": "ResNet18 (Binary Classifier)",
    "task": "Crack Detection",
    "classes": CLASSES_CRACK,
    "best_accuracy": float(best_acc),
    "test_accuracy": float(test_acc) if test_loader else None,
    "hyperparameters": {
        "lr": CRACK_LR, "weight_decay": CRACK_WEIGHT_DECAY,
        "epochs": CRACK_NUM_EPOCHS, "patience": CRACK_PATIENCE,
        "dropout": 0.5, "augmentation": "light (flip, rotate, affine, colorjitter)"
    },
    "datasets_used": ["Smartphone_Screen_Damage_Detection"],
    "train_size": len(train_ds),
    "val_size": len(valid_ds) if valid_ds else 0,
    "test_size": len(test_ds) if test_ds else 0,
    "test_split_ratio": "pre-split (train/valid/test folders)",
}
(RESULTS_DIR / "crack_detector_results.json").write_text(json.dumps(results, indent=2))
print("\nDone: Crack detector.")

Crack — Train: 395 | Valid: 100 | Test: 24
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 176MB/s] 
Training: 100%|██████████| 13/13 [00:18<00:00,  1.45s/it]


Epoch 1/15 — Train Loss: 0.2212 Acc: 0.8987 | Val Loss: 3.2381 Acc: 0.5800


Training: 100%|██████████| 13/13 [00:15<00:00,  1.18s/it]


Epoch 2/15 — Train Loss: 0.1099 Acc: 0.9595 | Val Loss: 0.2880 Acc: 0.9400


...

Epoch 4/15 — Train Loss: 0.0736 Acc: 0.9747 | Val Loss: 0.1529 Acc: 0.9600


Training: 100%|██████████| 13/13 [00:15<00:00,  1.15s/it]


Epoch 5/15 — Train Loss: 0.0521 Acc: 0.9823 | Val Loss: 0.0612 Acc: 0.9600


Training: 100%|██████████| 13/13 [00:14<00:00,  1.14s/it]


Epoch 6/15 — Train Loss: 0.0175 Acc: 0.9975 | Val Loss: 0.1266 Acc: 0.9400
Early stopping at epoch 6

Test Accuracy: 0.9583
              precision    recall  f1-score   support

 not_cracked       0.92      1.00      0.96        12
     cracked       1.00      0.92      0.96        12

    accuracy                           0.96        24
   macro avg       0.96      0.96      0.96        24
weighted avg       0.96      0.96      0.96        24


Done: Crack detector.


---
## 4. Corrosion Detector (Image — 5 Classes)

ResNet18 multi-class classifier: corrosion levels 5–9

Uses aggressive augmentation + label smoothing + lower LR.

Dataset: Corrosion dataset

In [6]:
# === Corrosion Detector: per-model config ===
CORROSION_IMG_SIZE = 224
CORROSION_BATCH_SIZE = 32
CORROSION_NUM_EPOCHS = 30
CORROSION_LR = 0.0005
CORROSION_WEIGHT_DECAY = 1e-4
CORROSION_PATIENCE = 5

# Aggressive augmentation for small corrosion dataset
corrosion_train_transform = transforms.Compose([
    transforms.Resize((CORROSION_IMG_SIZE, CORROSION_IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.15, 0.15), scale=(0.85, 1.15)),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.1)),
])

corrosion_val_transform = transforms.Compose([
    transforms.Resize((CORROSION_IMG_SIZE, CORROSION_IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class CorrosionDetector(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        self.backbone = models.resnet18(weights="DEFAULT")
        self.backbone.fc = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(self.backbone.fc.in_features, num_classes),
        )
    def forward(self, x): return self.backbone(x)

# Load datasets
CORROSION_ROOT = CORROSION_DATA
CLASSES_CORROSION = ["5", "6", "7", "8", "9"]
CV_SPLIT = "cross_val_1"

train_path = CORROSION_ROOT / CV_SPLIT / "train"
valid_path = CORROSION_ROOT / CV_SPLIT / "val"

train_ds = datasets.ImageFolder(str(train_path), transform=corrosion_train_transform)
valid_ds = datasets.ImageFolder(str(valid_path), transform=corrosion_val_transform) if valid_path.exists() else None
print(f"Corrosion — Train: {len(train_ds)} | Valid: {len(valid_ds) if valid_ds else 0}")

train_loader = DataLoader(train_ds, batch_size=CORROSION_BATCH_SIZE, shuffle=True, num_workers=0)
valid_loader = DataLoader(valid_ds, batch_size=CORROSION_BATCH_SIZE, shuffle=False, num_workers=0) if valid_ds else None

model = CorrosionDetector().to(DEVICE)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.Adam(model.parameters(), lr=CORROSION_LR, weight_decay=CORROSION_WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CORROSION_NUM_EPOCHS)

best_acc = 0.0
patience_counter = 0
t_losses, t_accs, v_losses, v_accs = [], [], [], []

for epoch in range(CORROSION_NUM_EPOCHS):
    tl, ta = img_train_epoch(model, train_loader, criterion, optimizer, DEVICE)
    t_losses.append(tl); t_accs.append(ta)
    msg = f"Epoch {epoch+1}/{CORROSION_NUM_EPOCHS} — Train Loss: {tl:.4f} Acc: {ta:.4f}"
    if valid_loader:
        vl, va, _, _ = img_val_epoch(model, valid_loader, criterion, DEVICE)
        v_losses.append(vl); v_accs.append(va)
        msg += f" | Val Loss: {vl:.4f} Acc: {va:.4f}"
        if va > best_acc:
            best_acc = va
            patience_counter = 0
            torch.save(model.state_dict(), MODEL_DIR / "corrosion_detector.pth")
        else:
            patience_counter += 1
    print(msg)
    scheduler.step()
    if patience_counter >= CORROSION_PATIENCE:
        print(f"Early stopping at epoch {epoch+1}")
        break

# Load best and evaluate
model.load_state_dict(torch.load(MODEL_DIR / "corrosion_detector.pth", map_location=DEVICE, weights_only=True))
if valid_loader:
    _, val_acc, val_preds, val_labels = img_val_epoch(model, valid_loader, criterion, DEVICE)
    print(f"\nValidation Accuracy: {val_acc:.4f}")
    print(classification_report(val_labels, val_preds, target_names=CLASSES_CORROSION))

    plot_confusion_matrix(val_labels, val_preds, CLASSES_CORROSION,
        RESULTS_DIR / "corrosion_detector_confusion_matrix.png", "Corrosion Detector — Confusion Matrix")
    plot_training_curves(t_losses, v_losses, t_accs, v_accs,
        RESULTS_DIR / "corrosion_detector_training_curves.png")
    plot_per_class_pr(val_labels, val_preds, CLASSES_CORROSION,
        RESULTS_DIR / "corrosion_detector_per_class_pr.png", "Corrosion Detector — Per-Class P&R")

    model.eval()
    all_scores = []
    with torch.no_grad():
        for images, _ in valid_loader:
            all_scores.extend(torch.softmax(model(images.to(DEVICE)), dim=1).cpu().numpy())
    plot_roc_auc(val_labels, np.array(all_scores), CLASSES_CORROSION,
        RESULTS_DIR / "corrosion_detector_roc_auc.png", "Corrosion Detector — ROC Curve")

results = {
    "model_type": "ResNet18 (Multi-Class Classifier)",
    "task": "Corrosion Level Detection",
    "classes": CLASSES_CORROSION,
    "best_accuracy": float(best_acc),
    "val_accuracy": float(val_acc) if valid_loader else None,
    "hyperparameters": {
        "lr": CORROSION_LR, "weight_decay": CORROSION_WEIGHT_DECAY,
        "epochs": CORROSION_NUM_EPOCHS, "patience": CORROSION_PATIENCE,
        "dropout": 0.3, "label_smoothing": 0.1,
        "augmentation": "aggressive (flip, rotate, colorjitter, affine, perspective, erasing)"
    },
    "datasets_used": ["corrosion"],
    "train_size": len(train_ds),
    "val_size": len(valid_ds) if valid_ds else 0,
    "val_split_ratio": "pre-split (cross_val_1)",
}
(RESULTS_DIR / "corrosion_detector_results.json").write_text(json.dumps(results, indent=2))
print("\nDone: Corrosion detector.")

Corrosion — Train: 479 | Valid: 61


Training: 100%|██████████| 15/15 [00:09<00:00,  1.65it/s]


Epoch 1/30 — Train Loss: 1.3835 Acc: 0.4823 | Val Loss: 1.6108 Acc: 0.5574


Training: 100%|██████████| 15/15 [00:07<00:00,  2.11it/s]


Epoch 2/30 — Train Loss: 1.1645 Acc: 0.5908 | Val Loss: 2.3207 Acc: 0.4754


...

Epoch 10/30 — Train Loss: 0.8890 Acc: 0.7432 | Val Loss: 0.9616 Acc: 0.7213


Training: 100%|██████████| 15/15 [00:07<00:00,  2.07it/s]


Epoch 11/30 — Train Loss: 0.9288 Acc: 0.7077 | Val Loss: 1.0189 Acc: 0.6721


Training: 100%|██████████| 15/15 [00:07<00:00,  2.04it/s]


Epoch 12/30 — Train Loss: 0.9320 Acc: 0.7349 | Val Loss: 0.9649 Acc: 0.7049
Early stopping at epoch 12

Validation Accuracy: 0.7541
              precision    recall  f1-score   support

           5       1.00      0.58      0.74        12
           6       0.62      0.62      0.62        13
           7       0.65      0.92      0.76        12
           8       0.77      0.83      0.80        12
           9       0.91      0.83      0.87        12

    accuracy                           0.75        61
   macro avg       0.79      0.76      0.76        61
weighted avg       0.79      0.75      0.75        61


Done: Corrosion detector.


---
## 5. Component Classifier (Image — 10 Classes)

ResNet18 multi-class classifier for laptop components

Dataset: Laptop Components Image Dataset

In [9]:
# === Component Classifier: per-model config ===
COMP_IMG_SIZE = 224
COMP_BATCH_SIZE = 32
COMP_NUM_EPOCHS = 10
COMP_LR = 0.001
COMP_WEIGHT_DECAY = 1e-4
COMP_MIN_DELTA=0.005
COMP_PATIENCE = 2


# Standard augmentation for larger dataset
comp_train_transform = transforms.Compose([
    transforms.Resize((COMP_IMG_SIZE, COMP_IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

comp_val_transform = transforms.Compose([
    transforms.Resize((COMP_IMG_SIZE, COMP_IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class ComponentClassifier(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.backbone = models.resnet18(weights="DEFAULT")
        self.backbone.fc = nn.Linear(self.backbone.fc.in_features, num_classes)
    def forward(self, x): return self.backbone(x)

class ComponentDataset(Dataset):
    def __init__(self, paths, labels, transform=None):
        self.paths, self.labels, self.transform = paths, labels, transform
    def __len__(self): return len(self.paths)
    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        if self.transform: img = self.transform(img)
        return img, self.labels[idx]

# Discover images
COMP_ROOT = COMPONENT_DATA
COMPONENTS = ["Battery", "LCDScreen", "Keyboard", "Hinge", "Motherboard",
              "HardDiskDrive", "RAM", "Processor", "WebCam", "TouchPad"]

component_map = {}
for idx, comp in enumerate(COMPONENTS):
    for folder in COMP_ROOT.iterdir():
        if folder.is_dir() and comp.lower() in folder.name.lower():
            component_map[idx] = folder
            break

paths, labels = [], []
for label_idx, folder in component_map.items():
    for ext in ("*.jpg", "*.jpeg", "*.png"):
        for img_path in folder.glob(ext):
            paths.append(str(img_path))
            labels.append(label_idx)
print(f"Component dataset: {len(paths)} images across {len(component_map)} classes")

train_paths, val_paths, train_labels, val_labels = train_test_split(
    paths, labels, test_size=0.2, random_state=42, stratify=labels)

train_loader = DataLoader(ComponentDataset(train_paths, train_labels, comp_train_transform),
    batch_size=COMP_BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(ComponentDataset(val_paths, val_labels, comp_val_transform),
    batch_size=COMP_BATCH_SIZE, shuffle=False, num_workers=0)

model = ComponentClassifier(len(COMPONENTS)).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=COMP_LR, weight_decay=COMP_WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=COMP_NUM_EPOCHS)

best_acc = 0.0
patience_counter = 0
t_losses, t_accs, v_losses, v_accs = [], [], [], []

for epoch in range(COMP_NUM_EPOCHS):
    tl, ta = img_train_epoch(model, train_loader, criterion, optimizer, DEVICE)
    vl, va, _, _ = img_val_epoch(model, val_loader, criterion, DEVICE)
    t_losses.append(tl); t_accs.append(ta); v_losses.append(vl); v_accs.append(va)
    print(f"Epoch {epoch+1}/{COMP_NUM_EPOCHS} — Train Loss: {tl:.4f} Acc: {ta:.4f} | Val Loss: {vl:.4f} Acc: {va:.4f}")
    if va > best_acc + COMP_MIN_DELTA:
        best_acc = va
        patience_counter = 0
        torch.save(model.state_dict(), MODEL_DIR / "image_classifier_laptop.pth")
    else:
        patience_counter += 1
    scheduler.step()
    if patience_counter >= COMP_PATIENCE:
        print(f"Early stopping at epoch {epoch+1}")
        break

# Final evaluation
model.load_state_dict(torch.load(MODEL_DIR / "image_classifier_laptop.pth", map_location=DEVICE, weights_only=True))
_, test_acc, test_preds, test_labels = img_val_epoch(model, val_loader, criterion, DEVICE)
print(f"\nValidation Accuracy: {test_acc:.4f}")
print(classification_report(test_labels, test_preds, target_names=COMPONENTS))

plot_confusion_matrix(test_labels, test_preds, COMPONENTS,
    RESULTS_DIR / "component_classifier_confusion_matrix.png", "Component Classifier — Confusion Matrix")
plot_training_curves(t_losses, v_losses, t_accs, v_accs,
    RESULTS_DIR / "component_classifier_training_curves.png")
plot_per_class_pr(test_labels, test_preds, COMPONENTS,
    RESULTS_DIR / "component_classifier_per_class_pr.png", "Component Classifier — Per-Class P&R")

all_scores = []
model.eval()
with torch.no_grad():
    for images, _ in val_loader:
        all_scores.extend(torch.softmax(model(images.to(DEVICE)), dim=1).cpu().numpy())
plot_roc_auc(test_labels, np.array(all_scores), COMPONENTS,
    RESULTS_DIR / "component_classifier_roc_auc.png", "Component Classifier — ROC Curve")

results = {
    "model_type": "ResNet18 (Multi-Class Classifier)",
    "task": "Laptop Component Classification",
    "classes": COMPONENTS,
    "best_accuracy": float(best_acc),
    "test_accuracy": float(test_acc),
    "datasets_used": ["Laptop Components Image Dataset to Classify Different Components"],
    "train_size": len(train_paths),
    "val_size": len(val_paths),
    "val_split_ratio": 0.2,
}
(RESULTS_DIR / "component_classifier_results.json").write_text(json.dumps(results, indent=2))
print("\nDone: Component classifier.")

Component dataset: 9800 images across 10 classes


Training: 100%|██████████| 245/245 [02:25<00:00,  1.68it/s]


Epoch 1/10 — Train Loss: 0.6378 Acc: 0.7760 | Val Loss: 0.6496 Acc: 0.7918


Training: 100%|██████████| 245/245 [02:21<00:00,  1.73it/s]


Epoch 2/10 — Train Loss: 0.3354 Acc: 0.8889 | Val Loss: 0.4617 Acc: 0.8587


...

Epoch 7/10 — Train Loss: 0.0552 Acc: 0.9812 | Val Loss: 0.0170 Acc: 0.9929


Training: 100%|██████████| 245/245 [02:20<00:00,  1.75it/s]


Epoch 8/10 — Train Loss: 0.0281 Acc: 0.9909 | Val Loss: 0.0140 Acc: 0.9949


Training: 100%|██████████| 245/245 [02:19<00:00,  1.76it/s]


Epoch 9/10 — Train Loss: 0.0220 Acc: 0.9944 | Val Loss: 0.0122 Acc: 0.9954
Early stopping at epoch 9

Validation Accuracy: 0.9929
               precision    recall  f1-score   support

      Battery       0.98      0.98      0.98       196
    LCDScreen       0.99      0.99      0.99       196
     Keyboard       1.00      0.99      0.99       196
        Hinge       1.00      0.99      1.00       196
  Motherboard       1.00      1.00      1.00       196
HardDiskDrive       1.00      1.00      1.00       196
          RAM       0.99      0.99      0.99       196
    Processor       0.98      0.99      0.99       196
       WebCam       0.99      1.00      1.00       196
     TouchPad       0.98      0.99      0.98       196

     accuracy                           0.99      1960
    macro avg       0.99      0.99      0.99      1960
 weighted avg       0.99      0.99      0.99      1960


Done: Component classifier.


---
## 5b. Image Models Complete

All image models are trained and saved. To download:
1. Click **Output** in the right sidebar
2. Find `training/results/` and `models/` folders
3. Click the download icon next to each file

Files saved:
- `models/crack_detector.pth`
- `models/corrosion_detector.pth`
- `models/image_classifier_laptop.pth`
- `training/results/*.json` and `*.png`

---
## 6. Summary

In [11]:
print("=" * 70)
print("TRAINING SUMMARY")
print("=" * 70)

# List all result files
for f in sorted(glob.glob(str(RESULTS_DIR / "*.json"))):
    with open(f) as fp:
        data = json.load(fp)
    name = Path(f).stem
    print(f"\n--- {name} ---")
    print(json.dumps(data, indent=2))

print("\n" + "=" * 70)
print("Saved models:")
for f in sorted(glob.glob(str(MODEL_DIR / "*"))):
    size = os.path.getsize(f)
    print(f"  {Path(f).name} ({size / 1024 / 1024:.1f} MB)")

print("\nSaved visualizations:")
for f in sorted(glob.glob(str(RESULTS_DIR / "*.png"))):
    size = os.path.getsize(f)
    print(f"  {Path(f).name} ({size / 1024:.1f} KB)")

print("\n" + "=" * 70)
print("All 5 models trained successfully!")
print("=" * 70)

TRAINING SUMMARY

--- component_classifier_results ---
{
  "model_type": "ResNet18 (Multi-Class Classifier)",
  "task": "Laptop Component Classification",
  "classes": [
    "Battery",
    "LCDScreen",
    "Keyboard",
    "Hinge",
    "Motherboard",
    "HardDiskDrive",
    "RAM",
    "Processor",
    "WebCam",
    "TouchPad"
  ],
  "best_accuracy": 0.9928571428571429,
  "test_accuracy": 0.9928571428571429,
  "datasets_used": [
    "Laptop Components Image Dataset to Classify Different Components"
  ],
  "train_size": 7840,
  "val_size": 1960,
  "val_split_ratio": 0.2
}

--- corrosion_detector_results ---
{
  "model_type": "ResNet18 (Multi-Class Classifier)",
  "task": "Corrosion Level Detection",
  "classes": [
    "5",
    "6",
    "7",
    "8",
    "9"
  ],
  "best_accuracy": 0.7540983606557377,
  "val_accuracy": 0.7540983606557377,
  "hyperparameters": {
    "lr": 0.0005,
    "weight_decay": 0.0001,
    "epochs": 30,
    "patience": 5,
    "dropout": 0.3,
    "label_smoothing": 0.1